# **Datalab II Sprint 3 2025 - 2026**
**Plaats:** De Haagse Hogenschool, ADS & AI<br>
**Auteurs:** M. Kilinc, D. Hoogenbosch, J. Wolthuis, S. Slingerland, L. van Hamersveld<br>
**Groep:** B2 <br>
**Coach:** Onur Tezel <br>
**Datum:** 31/03/2026


| Naam  | Studentnummer |
|-------|---------------|
| Lucas | 25076116      |
| Sandro| 25154370      |
| Mehmet| 25135007      |
| Julius| 25090216      |
| Dylan | 25118498      |
---

## **Inhoudsopgave**
1. [Imports & Configuratie](#1)
2. [Data inladen](#2)
3. [Data Exploration met SQL](#3)
4. [Onderzoeken Teameigenschappen](#4)

---
<a id='1'></a>
## **1. Imports & Configuratie**

In [1]:
# Imports
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from scipy.stats import pearsonr, f_oneway

In [2]:
# Via deze functie in pandas zorgen we ervoor dat we de DataFrames volledig kunnen weergeven in de cell
pd.set_option('display.max_rows', 200)

In [3]:
# Herbruikbare functie om een SQL-query uit te voeren en het resultaat te tonen
def voer_query_uit(query, conn, toon=True):
    """Voert een SQL-query uit en geeft het resultaat terug als DataFrame.
    
    Parameters:
        query (str): De SQL-query om uit te voeren.
        conn: De database connectie.
        toon (bool): Als True wordt het DataFrame getoond. Standaard True.
    """
    df = pd.read_sql(query, conn)
    if toon:
        display(df)
    return df

---
<a id='2'></a>
## **2. Data inladen**
In de onderstaande kolommen laden we de data op dezelfde manier als de vorige keer, namelijk via de Kaggle API. <br> 
Hieronder vind je een uitleg over het gebruik en een referentie.



### **2.1 Gebruik van de Kaggle API** 
We gebruiken de Kaggle API om de data lokaal op te slaan en dit hoeft slechts eenmalig te gebeuren.<br>Download de API-sleutel uit onze GitHub-repository en plaats deze in de map C:\Users\<Jouw_gebruikersnaam>\.kaggle. <br>

Raadpleeg voor verdere instructies de officiële documentatie op https://www.kaggle.com/docs/api.<br>
Verwijder eventueel het # in het script, voer de pip install en de data-import uit en je bent klaar om de data lokaal te gebruiken.


In [4]:
# %pip install kaggle 
import kaggle 
kaggle.api.authenticate()
kaggle.api.dataset_download_files("hugomathien/soccer", path='.', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer


OSError: [Errno 30] Read-only file system: './soccer.zip'

In [ ]:
# Maak een verbinding met de SQLite-database "database.sqlite"
conn = sqlite3.connect("database.sqlite")

# Voer een SQL-query uit om alle gegevens uit de tabel 'Match' op te halen, het resultaat wordt ingelezen als pandas DataFrame
df = pd.read_sql("""

    SELECT *
        FROM Match

    """,
    conn)

---
<a id='4'></a>
## **4. Onderzoeken Teameigenschappen**

---
### 4.1 Teameigenschappen samenvoegen

In [ ]:
# query om teameigenschappen te selecteren en te groeperen op teamnaam
query = """
SELECT
    T.team_long_name AS Team,
    AVG(TA.buildUpPlaySpeed)            AS buildUpPlaySpeed,
    AVG(TA.buildUpPlayDribbling)        AS buildUpPlayDribbling,
    AVG(TA.buildUpPlayPassing)          AS buildUpPlayPassing,
    AVG(TA.chanceCreationPassing)       AS chanceCreationPassing,
    AVG(TA.chanceCreationCrossing)      AS chanceCreationCrossing,
    AVG(TA.chanceCreationShooting)      AS chanceCreationShooting,
    AVG(TA.defencePressure)             AS defencePressure,
    AVG(TA.defenceAggression)           AS defenceAggression,
    AVG(TA.defenceTeamWidth)            AS defenceTeamWidth
FROM Team_Attributes AS TA
JOIN Team AS T ON TA.team_api_id = T.team_api_id
GROUP BY T.team_long_name;
"""

# Opslaan data uit query in DataFrame
df_attrs = voer_query_uit(query, conn, toon=False)

# Query om team punten te berekenen
punten_query = """
SELECT
    T.team_long_name AS Team,
    COUNT(*) as Wedstrijden,
    SUM(CASE
        WHEN M.home_team_api_id = T.team_api_id AND M.home_team_goal > M.away_team_goal THEN 3
        WHEN M.away_team_api_id = T.team_api_id AND M.away_team_goal > M.home_team_goal THEN 3
        WHEN M.home_team_api_id = T.team_api_id AND M.home_team_goal = M.away_team_goal THEN 1
        WHEN M.away_team_api_id = T.team_api_id AND M.home_team_goal = M.away_team_goal THEN 1
        ELSE 0
    END) as Punten
FROM Team AS T
JOIN Match AS M ON (M.home_team_api_id = T.team_api_id OR M.away_team_api_id = T.team_api_id)
GROUP BY T.team_long_name
ORDER BY Punten DESC;
"""

# Opslaan punten data in DataFrame
df_2 = voer_query_uit(punten_query, conn, toon=False)

# Samenvoegen met het punten DataFrame op teamnaam
df_merged = pd.merge(df_2, df_attrs, on='Team', how='inner')

# DataFrame met alle teameigenschappen en punten per team
display(df_merged.head(10))

# Print grote van de samengevoegde DataFrame
print(f'Samengevoegd DataFrame: {len(df_merged)} rijen, {len(df_merged.columns)} kolommen')

,Team,season,totaal_punten,klassering,buildUpPlaySpeed,buildUpPlayDribbling,buildUpPlayPassing,chanceCreationPassing,chanceCreationCrossing,chanceCreationShooting,defencePressure,defenceAggression,defenceTeamWidth
0,PSV,2015/2016,84,1,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
1,PSV,2014/2015,88,1,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
2,PSV,2013/2014,59,4,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
3,PSV,2012/2013,69,2,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
4,PSV,2011/2012,69,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
5,PSV,2010/2011,69,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
6,PSV,2009/2010,78,3,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
7,PSV,2008/2009,65,4,52.333333,39.0,38.333333,50.166667,54.5,54.000000,43.833333,41.500000,48.833333
8,Ajax,2015/2016,82,2,35.166667,41.5,33.833333,50.666667,58.5,49.666667,59.833333,53.833333,54.333333
9,Ajax,2014/2015,71,2,35.166667,41.5,33.833333,50.666667,58.5,49.666667,59.833333,53.833333,54.333333


Samengevoegd DataFrame: 143 rijen, 13 kolommen


In [ ]:
# Database connection is already established above as 'conn'

---

In [10]:

def bereken_speler_gemiddelden(connection):
    """
    Berekent gemiddelde attributen en identificeert keepers op basis van gk_reflexes.
    """
    query = """
    SELECT
        P.player_name AS Player,
        AVG(PA.overall_rating) AS avg_overall_rating,
        AVG(PA.potential) AS avg_potential,
        AVG(PA.finishing) AS avg_finishing,
        AVG(PA.dribbling) AS avg_dribbling,
        AVG(PA.sprint_speed) AS avg_sprint_speed,
        AVG(PA.stamina) AS avg_stamina,
        AVG(PA.gk_reflexes) AS avg_gk_reflexes
    FROM Player_Attributes AS PA
    JOIN Player AS P ON PA.player_api_id = P.player_api_id
    GROUP BY P.player_name;
    """
    # Data inladen
    df = pd.read_sql(query, connection)

    # Functie om rol te bepalen: als reflexen veel hoger zijn dan finishing, is het een keeper
    # Een drempelwaarde van 30-40 voor reflexen werkt meestal perfect voor deze dataset
    def bepaal_rol(row):
        gk_reflexes = row['avg_gk_reflexes'] if pd.notna(row['avg_gk_reflexes']) else 0
        finishing = row['avg_finishing'] if pd.notna(row['avg_finishing']) else 0
        if gk_reflexes > 30 and gk_reflexes > finishing:
            return 'Keeper'
        else:
            return 'Veldspeler'

    df['Rol'] = df.apply(bepaal_rol, axis=1)

    # Optioneel: Sorteer op de beste spelers
    df = df.sort_values(by='avg_overall_rating', ascending=False)

    return df

In [11]:
# Uitvoeren
df_resultaat = bereken_speler_gemiddelden(conn)

# Check de top 10 keepers
display(df_resultaat.head(50))

NameError: name 'conn' is not defined